In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import lightgbm as lgb

In [2]:
DATASET_PATH = "train_cropped/"
SEASON_LABELS = {"autunno": 0, "estate": 1, "inverno": 2, "primavera": 3}

In [3]:
X = []
y = []

In [4]:
def get_average_color(image_path):

    if not os.path.exists(image_path):
        return np.array([0, 0, 0])

    image = cv2.imread(image_path)

    if image is None:
        return np.array([0, 0, 0])

    image_hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    avg_color = np.mean(image_hsv.reshape(-1, 3), axis=0)
    return avg_color  # [H, S, V]

In [5]:
file_names = []
for season in os.listdir(DATASET_PATH):
    season_path = os.path.join(DATASET_PATH, season)

    if not os.path.isdir(season_path) or season not in SEASON_LABELS:
        continue

    for subtype in os.listdir(season_path):
        subtype_path = os.path.join(season_path, subtype)

        if not os.path.isdir(subtype_path):
            continue

        hair_path = os.path.join(subtype_path, "hair")
        right_eye_path = os.path.join(subtype_path, "r_eye")
        left_eye_path = os.path.join(subtype_path, "l_eye")
        skin_path = os.path.join(subtype_path, "face")

        for file in os.listdir(skin_path):
            if file.endswith(".jpg") or file.endswith(".png"):
                file_name = file[:-15]
                print(file_name)

                skin_img = os.path.join(skin_path, f"{file_name}_face_color.png")
                hair_img = os.path.join(hair_path, f"{file_name}_hair_color.png")
                right_eye_img = os.path.join(right_eye_path, f"{file_name}_r_eye_color.png")
                left_eye_img = os.path.join(left_eye_path, f"{file_name}_l_eye_color.png")

                skin_color = get_average_color(skin_img)
                hair_color = get_average_color(hair_img)
                right_eye_color = get_average_color(right_eye_img)
                left_eye_color = get_average_color(left_eye_img)
                eyes_color = np.mean([left_eye_color, right_eye_color], axis = 0)

                X.append(np.concatenate([skin_color, hair_color, eyes_color]))
                y.append(SEASON_LABELS[season])
                file_names.append(file_name)

10063
10306
10411
10552
10748
10888
11113
11174
11199
11409
11438
11764
12016
12062
12109
1210
12148
12178
12251
12386
12497
12620
12769
12795
12887
12897
12962
12982
12992
13081
13212
13232
13339
13450
13478
13479
13533
13685
13721
13933
14207
14305
14374
14515
14520
14706
14841
14916
14951
15000
15027
15060
15102
15205
15219
15270
15320
1541
15420
15428
15503
15559
15685
15882
15921
15924
16006
16336
16444
16551
16836
16858
1692
17091
17164
17217
17265
17294
17677
17684
17712
17760
17866
179
18122
18258
18390
18495
18567
18577
18661
18687
18716
18731
18750
18777
18803
18880
18936
19000
19130
19169
19324
19653
19721
19827
19908
19987
20118
20202
2025
20526
2110
21720
21793
21853
2185
21895
22270
22290
22320
22408
22821
22889
23139
23232
23298
23366
2342
23712
23786
23839
2409
24247
24577
24859
25041
25267
2528
25386
26168
26182
26361
26459
2661
26689
26740
26962
27253
27425
2765
27673
27677
27758
27791
27871
28031
28148
28319
28451
28490
28593
2880
28885
28886
29014
29029
29542
2957
2

In [6]:
X = np.array(X)
y = np.array(y)

df = pd.DataFrame(X, columns=["skin_H", "skin_S", "skin_V", "hair_H", "hair_S", "hair_V", "eyes_H", "eyes_S", "eyes_V"])
df['file_name'] = file_names
df["season"] = y
df.to_csv("train_df.csv", index=False)

In [11]:
df = df.drop(columns=['file_name'])

In [12]:
X_train = df.drop(columns=['season'])
y_train = df['season']

In [14]:
X_train

,skin_H,skin_S,skin_V,hair_H,hair_S,hair_V,eyes_H,eyes_S,eyes_V
0,2.735214,35.176205,38.397236,6.118050,70.531616,22.145611,0.167196,0.275297,0.231951
1,3.777584,35.856396,47.519199,11.926365,26.448215,10.500534,0.045797,0.443148,0.252542
2,2.092743,41.345207,35.195572,7.993881,76.921268,17.644547,0.034502,0.315285,0.156099
3,2.797218,30.818752,55.316090,3.212650,4.931763,6.429718,0.058933,0.166666,0.150980
4,3.745140,54.939365,37.101215,0.599487,3.102638,0.905834,0.027132,0.358675,0.190672
...,...,...,...,...,...,...,...,...,...
4003,0.726457,6.670281,10.924288,5.328641,42.031360,22.637984,0.018389,0.091926,0.088957
4004,1.323478,12.309784,22.665031,11.448461,56.082328,55.224741,0.044241,0.108758,0.117124
4005,1.104423,9.423651,19.986334,3.151537,32.071567,23.425926,0.019821,0.128713,0.163579
4006,3.076880,12.888285,33.340302,5.596239,48.526189,44.051886,0.052031,0.151235,0.201647


In [26]:
DATASET_PATH = "test_cropped/"

In [27]:
X = []
y = []

In [28]:
file_names = []
for season in os.listdir(DATASET_PATH):
    season_path = os.path.join(DATASET_PATH, season)

    if not os.path.isdir(season_path) or season not in SEASON_LABELS:
        continue

    hair_path = os.path.join(season_path, "hair")
    right_eye_path = os.path.join(season_path, "r_eye")
    left_eye_path = os.path.join(season_path, "l_eye")
    skin_path = os.path.join(season_path, "face")

    for file in os.listdir(skin_path):
        if file.endswith(".jpg") or file.endswith(".png"):
            file_name = file[:-15]
            print(file_name)

            skin_img = os.path.join(skin_path, f"{file_name}_face_color.png")
            hair_img = os.path.join(hair_path, f"{file_name}_hair_color.png")
            right_eye_img = os.path.join(right_eye_path, f"{file_name}_r_eye_color.png")
            left_eye_img = os.path.join(left_eye_path, f"{file_name}_l_eye_color.png")

            skin_color = get_average_color(skin_img)
            hair_color = get_average_color(hair_img)
            right_eye_color = get_average_color(right_eye_img)
            left_eye_color = get_average_color(left_eye_img)
            eyes_color = np.mean([left_eye_color, right_eye_color], axis = 0)

            X.append(np.concatenate([skin_color, hair_color, eyes_color]))
            y.append(SEASON_LABELS[season])
            file_names.append(file_name)

10443
10528
10602
10962
10973
11102
11222
11564
11566
11719
11726
11802
11984
12486
12845
13209
1346
13498
13499
13716
13885
1402
14128
14183
14264
14618
14920
14
15026
15172
15178
15449
15701
15868
16022
1620
16295
16347
16378
16411
16725
168
16936
16962
17018
17203
17504
17696
17857
17876
17973
18176
18351
18389
1860
18644
18826
18882
19284
19413
19453
19518
19550
19572
1974
19762
19953
20008
20045
20115
2023
20420
20431
20455
20500
20541
20701
20823
20841
21062
21124
21280
21295
21368
21432
21438
21442
21518
21563
21578
21655
21657
21702
21731
21771
2178
21875
21977
22246
22292
22372
22389
22684
22717
22788
22886
22915
22919
22942
22979
23030
23031
23105
23189
23235
23341
23781
23813
23863
23902
24055
24073
24077
24219
24237
24239
24259
24345
24384
24530
24547
24549
24573
24692
24700
24703
24783
2483
24902
24906
24936
24982
24993
25099
25247
25266
25270
25289
25636
25637
25988
26032
26082
26323
26330
26377
26438
26511
26514
26598
26788
26801
26893
26969
26983
27281
27338
27382
27429

In [29]:
X = np.array(X)
y = np.array(y)

df_test = pd.DataFrame(X, columns=["skin_H", "skin_S", "skin_V", "hair_H", "hair_S", "hair_V", "eyes_H", "eyes_S", "eyes_V"])
df_test['file_name'] = file_names
df_test["season"] = y
df_test.to_csv("test_df.csv", index=False)

In [30]:
df_test = df_test.drop(columns=['file_name'])

In [31]:
X_new = df_test.drop(columns=['season'])
y_new = df_test['season']

In [32]:
X_valid, X_test, y_valid, y_test = train_test_split(X_new, y_new, test_size=0.5, random_state=42, stratify=y)